# DFT validation walkthrough (Co–Bi demo)

This notebook mirrors the CLI flow—select MLIP-shortlisted structures, write Quantum ESPRESSO pw.x inputs, then (optionally) parse scores—and adds **structure** and **formation-energy / hull** plots.

**Setup:** use a kernel from this repo's environment, and start Jupyter from the **repository root** (or adjust `REPO` below).

```bash
pip install -e ".[notebook]"
jupyter lab
```

Real QE pw.x still runs outside Python; after runs exist under `dft/runs/Co-Bi/<candidate_id>/`, use the parse cell or `scripts/parse_dft_results.py`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pymatgen.core import Composition, Structure

from hullgap.dft.dft_hull import (
    formation_energy_per_atom,
    hull_energy_at_x,
    load_elemental_references,
    lower_convex_hull_2d,
    score_dft_candidates,
    x_bi_fraction,
)
from hullgap.dft.make_qe_inputs import generate_inputs_from_candidate_list
from hullgap.dft.select_candidates import load_hull_scores_csv, select_top_candidates, write_candidate_list

REPO = Path.cwd().resolve()
if not (REPO / "data").is_dir() and (REPO.parent / "data").is_dir():
    REPO = REPO.parent
if not (REPO / "data").is_dir():
    raise FileNotFoundError(
        f"Expected data/ under {REPO}. Open the notebook from the hullgap repo root or set REPO manually."
    )

print("REPO =", REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

## 1. Select candidates (same logic as `scripts/select_dft_candidates.py`)

In [ ]:
hull_csv = REPO / "data/results/hull_scores_Co-Bi.csv"
relaxed_dir = REPO / "data/relaxed/Co-Bi"
candidate_list_path = REPO / "dft/results/dft_candidate_list.csv"
candidate_list_path.parent.mkdir(parents=True, exist_ok=True)

hull_df = load_hull_scores_csv(hull_csv)
selected = select_top_candidates(hull_df, relaxed_dir, top_n=10, max_atoms=40)
write_candidate_list(selected, candidate_list_path)
selected

## 2. Write QE pw.x inputs (`dft/inputs/Co-Bi/<id>/`)

In [ ]:
qe_out = REPO / "dft/inputs/Co-Bi"
qe_out.mkdir(parents=True, exist_ok=True)
generate_inputs_from_candidate_list(selected, qe_out, preset="coarse_relax", kppa=1000)
list(qe_out.iterdir())

## 3. Visualize the relaxed demo structure (fractional coordinates)

In [ ]:
cif_path = relaxed_dir / "candidate_demo_001.cif"
struct = Structure.from_file(cif_path)
species = [str(s.specie) for s in struct]
fc = struct.frac_coords

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
colors = {"Co": "#3366cc", "Bi": "#cc6633"}
for sym in set(species):
    mask = [s == sym for s in species]
    pts = fc[mask]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=120, label=sym, c=colors.get(sym, "#444444"))
ax.set_xlabel("x (frac)")
ax.set_ylabel("y (frac)")
ax.set_zlabel("z (frac)")
ax.set_title(f"Demo structure: {struct.composition.reduced_formula}")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Synthetic DFT energies + formation-energy / hull plot

After you have real QE pw.out runs, replace this block with `pd.read_csv(REPO / "dft/results/dft_energies_Co-Bi.csv")` from `parse_dft_results`.

In [ ]:
refs_path = REPO / "dft/reference_energies.yaml"
refs = load_elemental_references(refs_path)

synthetic = pd.DataFrame(
    [
        {"candidate_id": "demo_A", "formula": "Co3Bi", "status": "ok", "converged": True, "dft_energy_total_eV": -18.0},
        {"candidate_id": "demo_B", "formula": "CoBi", "status": "ok", "converged": True, "dft_energy_total_eV": -8.5},
        {"candidate_id": "demo_C", "formula": "CoBi3", "status": "ok", "converged": True, "dft_energy_total_eV": -22.0},
    ]
)
rows = []
for _, r in synthetic.iterrows():
    comp = Composition(str(r["formula"]))
    n = comp.num_atoms
    etot = float(r["dft_energy_total_eV"])
    rows.append(
        {
            **r.to_dict(),
            "dft_energy_per_atom_eV": etot / n,
            "n_atoms": n,
            "volume_per_atom": "",
            "total_magnetization": "",
            "final_structure_path": "",
            "error_message": "",
        }
    )

dft_df = pd.DataFrame(rows)
scored = score_dft_candidates(dft_df, "Co-Bi", refs)
scored

In [ ]:
hull_pts = [(0.0, 0.0), (1.0, 0.0)]
for _, r in dft_df.iterrows():
    if not r.get("converged"):
        continue
    comp = Composition(str(r["formula"]))
    etot = float(r["dft_energy_total_eV"])
    e_form = formation_energy_per_atom(etot, comp, refs)
    hull_pts.append((x_bi_fraction(comp), e_form))

hull_arr = lower_convex_hull_2d(np.array(hull_pts, dtype=float))
xs = np.linspace(0, 1, 200)
ys = np.array([hull_energy_at_x(hull_arr, x) for x in xs])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(xs, ys, color="#333", lw=2, label="Lower hull (formation E)")
for _, r in dft_df.iterrows():
    comp = Composition(str(r["formula"]))
    etot = float(r["dft_energy_total_eV"])
    e_form = formation_energy_per_atom(etot, comp, refs)
    xb = x_bi_fraction(comp)
    ax.scatter([xb], [e_form], s=80, zorder=5, label=r["candidate_id"])
ax.axhline(0.0, color="gray", ls="--", lw=1)
ax.set_xlabel(r"Mole fraction Bi ($x_{\mathrm{Bi}}$)")
ax.set_ylabel("Formation energy (eV / atom)")
ax.set_title("Co\u2013Bi demo: synthetic DFT totals + reference YAML")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 5. (Optional) Parse real QE output tree

Uncomment after `dft/runs/Co-Bi/<candidate_id>/pw.out` exists.

```python
from hullgap.dft.parse_qe_outputs import parse_run_tree, write_energy_table

run_root = REPO / "dft/runs/Co-Bi"
parsed = parse_run_tree(run_root)
write_energy_table(parsed, REPO / "dft/results/dft_energies_Co-Bi.csv")
parsed
```